In [ ]:
#LSTM vs MLP

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path

from tradinglab.data_feed import DataFeed
from tradinglab.features import build_dataset, train_test_split, N_FEATURES, to_sequences
from tradinglab.models import MLP, LSTMRegressor
from tradinglab.ml import train_model, predict

In [ ]:
# ---- 1. Discover the universe and load it ----
all_csvs = sorted(p.stem for p in Path('data/egx').glob('*.csv'))
print('files found:', all_csvs)

# egx30.csv is the INDEX, not a tradeable stock -- exclude it from the universe.
# It has a different schema (Date, Price, Open...) and is loaded separately,
# only as a benchmark, via data_feed.load_egx30_returns.
stock_symbols = [s for s in all_csvs if s.lower() != 'egx30']

feed = DataFeed.from_dir('data/egx', symbols=stock_symbols)
print('universe loaded:', feed.symbols)
print('date range:', feed.dates[0].date(), '->', feed.dates[-1].date(), f'({feed.n_days} days)')


In [ ]:
# ---- 2. Pick one stock ----
ticker = 'ABUK'                       # change to 'HRHO' or any symbol printed above
asset_idx = feed.symbols.index(ticker)

In [ ]:
# ---- 3. Build the dataset -- SINGLE LAG version, direct parallel to the sine wave ----
X_full, y = build_dataset(feed, asset_idx)   # X_full has 9 engineered features, in order
X = X_full[:, :1]                            # keep only column 0: "return" (today's return)

Xtr, ytr, Xte, yte = train_test_split(X, y)  # chronological 70/30 -- test IS the future

print(f'total samples: {len(X)}   train: {len(Xtr)}   test: {len(Xte)}')

In [ ]:
# ---- 4. Train -- same MLP shape as the lesson, done directly here ----
torch.manual_seed(0)
model = MLP(n_features=X.shape[1], hidden=32)
history = train_model(model, Xtr, ytr, Xte, yte, epochs=300, lr=0.01)
 
pred_train = predict(model, Xtr)
pred_test = predict(model, Xte)
 
# Baselines for context: "always predict zero return" and "predict no change"
# (i.e. tomorrow's return = today's return, using feature column 0).
zero_baseline = float(np.mean(yte**2))
persistence_baseline = float(np.mean((Xte[:, 0] - yte)**2))
 
print(f'final train loss: {history["train"][-1]:.6f}')
print(f'final test loss:  {history["test"][-1]:.6f}')
print(f'"predict zero" baseline:      {zero_baseline:.6f}')
print(f'"predict no change" baseline: {persistence_baseline:.6f}')

In [ ]:
# ---- 5. Plots: loss curves, and predicted vs actual for BOTH periods ----
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
 
axes[0].plot(history['train'], label='train loss')
axes[0].plot(history['test'], label='test loss')
axes[0].set_yscale('log'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[0].set_title(f'{ticker}: loss curves')
 
axes[1].plot(ytr, label='actual return', linewidth=1, alpha=.7)
axes[1].plot(pred_train, label='MLP prediction', linewidth=1, linestyle='--', alpha=.8)
axes[1].legend(); axes[1].grid(alpha=.3); axes[1].set_title('Training period: predicted vs actual')
 
axes[2].plot(yte, label='actual return', linewidth=1, alpha=.7)
axes[2].plot(pred_test, label='MLP prediction', linewidth=1, linestyle='--', alpha=.8)
axes[2].legend(); axes[2].grid(alpha=.3); axes[2].set_title('Testing period: predicted vs actual')
 
plt.tight_layout(); plt.show()

In [ ]:
# ---- 6. Accuracy Metrics 
def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))
 
def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))
 
# Baseline predictions, for the same comparison as the loss numbers above.
zero_pred_tr, zero_pred_te = np.zeros_like(ytr), np.zeros_like(yte)
persistence_pred_tr, persistence_pred_te = Xtr[:, 0], Xte[:, 0]
 
print(f"{'':22s}{'MAE':>12s}{'RMSE':>12s}")
print(f"{'MLP (train)':22s}{mae(pred_train, ytr):12.6f}{rmse(pred_train, ytr):12.6f}")
print(f"{'MLP (test)':22s}{mae(pred_test, yte):12.6f}{rmse(pred_test, yte):12.6f}")
print(f"{'predict zero (test)':22s}{mae(zero_pred_te, yte):12.6f}{rmse(zero_pred_te, yte):12.6f}")
print(f"{'predict no-change (test)':22s}{mae(persistence_pred_te, yte):12.6f}{rmse(persistence_pred_te, yte):12.6f}")

In [ ]:
# ---- 7. Now give the model a WINDOW instead of a single point ----
#
# Everything above used one number -- today's return -- to predict tomorrow's.
# Now we swap in an LSTM that sees the last `seq_len` days of returns instead
# of just today's. Same underlying column (`X`, single-lag returns), same
# train/test split, same number of training epochs -- the only thing that
# changes is how much recent history the model is allowed to see.
#
# `to_sequences` is applied to Xtr/Xte SEPARATELY (not to the full X before
# splitting), so no window straddles the train/test boundary and peeks into
# the future. Each split loses its first `seq_len` rows to warm-up, since a
# window needs that many prior rows that can't reach across the split.
 
# %%
seq_len = 5
 
Xtr_seq, ytr_seq = to_sequences(Xtr, ytr, seq_len)
Xte_seq, yte_seq = to_sequences(Xte, yte, seq_len)
 
print(f'single-lag:        train {len(Xtr)}   test {len(Xte)}')
print(f'{seq_len}-day window:      train {len(Xtr_seq)}   test {len(Xte_seq)}')

In [ ]:
# ---- 8. Train the LSTM ----
torch.manual_seed(0)
lstm = LSTMRegressor(n_features=X.shape[1], hidden=32)
lstm_history = train_model(lstm, Xtr_seq, ytr_seq, Xte_seq, yte_seq, epochs=300, lr=0.01)
 
lstm_pred_train = predict(lstm, Xtr_seq)
lstm_pred_test = predict(lstm, Xte_seq)
 
print(f'LSTM final train loss: {lstm_history["train"][-1]:.6f}')
print(f'LSTM final test loss:  {lstm_history["test"][-1]:.6f}')
print(f'MLP  final test loss:  {history["test"][-1]:.6f}   (for comparison, section 4)')

In [ ]:
# ---- 9. Plots: LSTM loss curves, and predicted vs actual for BOTH periods ----
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
 
axes[0].plot(lstm_history['train'], label='train loss')
axes[0].plot(lstm_history['test'], label='test loss')
axes[0].set_yscale('log'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[0].set_title(f'{ticker}: LSTM loss curves')
 
axes[1].plot(ytr_seq, label='actual return', linewidth=1, alpha=.7)
axes[1].plot(lstm_pred_train, label='LSTM prediction', linewidth=1, linestyle='--', alpha=.8)
axes[1].legend(); axes[1].grid(alpha=.3); axes[1].set_title('Training period: predicted vs actual')
 
axes[2].plot(yte_seq, label='actual return', linewidth=1, alpha=.7)
axes[2].plot(lstm_pred_test, label='LSTM prediction', linewidth=1, linestyle='--', alpha=.8)
axes[2].legend(); axes[2].grid(alpha=.3); axes[2].set_title('Testing period: predicted vs actual')
 
plt.tight_layout(); plt.show()

In [ ]:
# ---- 10. Did memory actually help? MLP vs LSTM, on the SAME test rows ----
#
# The MLP's section-6 numbers were computed on the full test set (yte). The
# LSTM's test set (yte_seq) is `seq_len` rows shorter (warm-up). To compare
# fairly, we re-score the MLP on that exact same reduced set of rows.
 
# %%
mlp_pred_test_matched = predict(model, Xte[seq_len:])
zero_pred_seq = np.zeros_like(yte_seq)
persistence_pred_seq = Xte[seq_len:, 0]
 
print(f"{'':22s}{'test loss':>12s}{'MAE':>12s}{'RMSE':>12s}")
print(f"{'MLP':22s}{nn.functional.mse_loss(torch.tensor(mlp_pred_test_matched), torch.tensor(yte[seq_len:])).item():12.6f}"
      f"{mae(mlp_pred_test_matched, yte[seq_len:]):12.6f}{rmse(mlp_pred_test_matched, yte[seq_len:]):12.6f}")
print(f"{'LSTM':22s}{lstm_history['test'][-1]:12.6f}{mae(lstm_pred_test, yte_seq):12.6f}{rmse(lstm_pred_test, yte_seq):12.6f}")
print(f"{'predict zero':22s}{'--':>12s}{mae(zero_pred_seq, yte_seq):12.6f}{rmse(zero_pred_seq, yte_seq):12.6f}")
print(f"{'predict no-change':22s}{'--':>12s}{mae(persistence_pred_seq, yte_seq):12.6f}{rmse(persistence_pred_seq, yte_seq):12.6f}")